In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary f00iles to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/takuok/glove840b300dtxt/glove.840B.300d.txt
/kaggle/input/notebooks/plpany/bag-of-words-meets-bags-of-popcorn-by-the-imdb/__results__.html
/kaggle/input/notebooks/plpany/bag-of-words-meets-bags-of-popcorn-by-the-imdb/__notebook__.ipynb
/kaggle/input/notebooks/plpany/bag-of-words-meets-bags-of-popcorn-by-the-imdb/__output__.json
/kaggle/input/notebooks/plpany/bag-of-words-meets-bags-of-popcorn-by-the-imdb/glove.840B.300d.gensim.txt
/kaggle/input/notebooks/plpany/bag-of-words-meets-bags-of-popcorn-by-the-imdb/custom.css
/kaggle/input/notebooks/plpany/bag-of-words-meets-bags-of-popcorn-by-the-imdb/pickle/imdb_glove.pickle3
/kaggle/input/notebooks/plpany/bag-of-words-meets-bags-of-popcorn-by-the-imdb/corpus/imdb/testData.tsv
/kaggle/input/notebooks/plpany/bag-of-words-meets-bags-of-popcorn-by-the-imdb/corpus/imdb/labeledTrainData.tsv
/kaggle/input/notebooks/plpany/bag-of-words-meets-bags-of-popcorn-by-the-imdb/result/attlstm.csv
/kaggle/input/notebooks/plpany/bag-of-

# Part A · 手写数据处理（5 小步，替代 imdb_process.py）

In [2]:
!mkdir -p corpus/imdb result
!cd corpus/imdb && unzip -o -q /kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip
!cd corpus/imdb && unzip -o -q /kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip

### 下载一些必要的包kaggle里面没有

In [3]:
!pip install gensim beautifulsoup4 lxml

## 导包

In [4]:
import re
import os
import pandas as pd
import torch
import gensim
from bs4 import BeautifulSoup
from itertools import chain
from sklearn.model_selection import train_test_split


## 导数据

In [5]:
train = pd.read_csv('corpus/imdb/labeledTrainData.tsv',quoting=3,header=0,delimiter='\t')
test = pd.read_csv('corpus/imdb/testData.tsv',quoting=3,header=0,delimiter='\t')

train.head() , test.head()

(         id  sentiment                                             review
 0  "5814_8"          1  "With all this stuff going down at the moment ...
 1  "2381_9"          1  "\"The Classic War of the Worlds\" by Timothy ...
 2  "7759_3"          0  "The film starts with a manager (Nicholas Bell...
 3  "3630_4"          0  "It must be assumed that those who praised thi...
 4  "9495_8"          1  "Superbly trashy and wondrously unpretentious ...,
            id                                             review
 0  "12311_10"  "Naturally in a film who's main themes are of ...
 1    "8348_2"  "This movie is a disaster within a disaster fi...
 2    "5828_4"  "All in all, this is a movie for kids. We saw ...
 3    "7186_2"  "Afraid of the Dark left me with the impressio...
 4   "12128_7"  "A very accurate depiction of small time mob l...)

## 清洗数据

In [6]:
def review_to_wordlist(review):
    text = BeautifulSoup(review,'html.parser').get_text()        # 去html
    text = re.sub(r'[^a-zA-Z]',' ',text)                        # 去非字母
    return text.lower().split()                                 # 返回的都是小写

In [7]:
clean_train = [review_to_wordlist(r) for r in train["review"]]
clean_test = [review_to_wordlist(r) for r in test["review"]]
clean_train[:5],clean_test[:5]

([['with',
   'all',
   'this',
   'stuff',
   'going',
   'down',
   'at',
   'the',
   'moment',
   'with',
   'mj',
   'i',
   've',
   'started',
   'listening',
   'to',
   'his',
   'music',
   'watching',
   'the',
   'odd',
   'documentary',
   'here',
   'and',
   'there',
   'watched',
   'the',
   'wiz',
   'and',
   'watched',
   'moonwalker',
   'again',
   'maybe',
   'i',
   'just',
   'want',
   'to',
   'get',
   'a',
   'certain',
   'insight',
   'into',
   'this',
   'guy',
   'who',
   'i',
   'thought',
   'was',
   'really',
   'cool',
   'in',
   'the',
   'eighties',
   'just',
   'to',
   'maybe',
   'make',
   'up',
   'my',
   'mind',
   'whether',
   'he',
   'is',
   'guilty',
   'or',
   'innocent',
   'moonwalker',
   'is',
   'part',
   'biography',
   'part',
   'feature',
   'film',
   'which',
   'i',
   'remember',
   'going',
   'to',
   'see',
   'at',
   'the',
   'cinema',
   'when',
   'it',
   'was',
   'originally',
   'released',
   'some',


In [8]:
train_labels = train["sentiment"].tolist()
train_labels[:5]

[1, 1, 0, 0, 1]

## 构建正反向字典

In [9]:
vocab = set(chain(*clean_train)) | set(chain(*clean_test))    # 并集  两个集合的所有词
word_to_idx = {word:i + 1 for i,word in enumerate(vocab)}
word_to_idx['<unk>'] = 0                                             # 未知词占位符
idx_to_word = {i:w for w,i in word_to_idx.items()}                 # 反向字典
print(len(vocab))

101399


In [10]:
max_len = 512

In [11]:
def encode_samples(tokenized): # 词列表 -> 索引列表     最后返回一个数字列表
    return [[word_to_idx.get(w,0) for w in sample] for sample in tokenized]

In [12]:
def pad_samples(features,maxlen=max_len,PAD=0): # 这个的作用是进行填补但是填补的是什么东西呢
    out = []                                    # 实际上是填补列表也就是数组
    for f in features:
        f = f[:maxlen]          # 过长截断
        f = f + [PAD] * (max_len - len(f))
        out.append(f)
    return out

In [13]:
tr_reviews , val_reviews, tr_labels, val_labels = train_test_split(clean_train,
                                                                   train_labels,
                                                                   test_size=0.2,
                                                                   random_state=0)

In [14]:

# tensor是什么东西 其实是一个将数字列表将其转换成向量的一个（应该说是张量）
train_features = torch.tensor(pad_samples(encode_samples(tr_reviews)))
val_features = torch.tensor(pad_samples(encode_samples(val_reviews)))
test_features = torch.tensor(pad_samples(encode_samples(clean_test)))
train_labels_t = torch.tensor(tr_labels)
val_labels_t = torch.tensor(val_labels)

## 加载 glove表

In [15]:
src = "/kaggle/input/datasets/takuok/glove840b300dtxt/glove.840B.300d.txt"
dst = "/kaggle/working/glove.840B.300d.gensim.txt"

In [16]:
if not os.path.exists(dst):
    with open(src,encoding='utf-8') as f:
        n = sum(1 for _ in f)
    with open(src,encoding='utf-8') as f,open(dst,"w",encoding='utf-8') as g:
        g.write(f"{n} 300\n")
        for line in f : g.write(line)


In [17]:
wvmodel = gensim.models.KeyedVectors.load_word2vec_format(dst,
                                                          binary=False,
                                                         unicode_errors='ignore')

In [18]:
weight = torch.zeros(len(vocab)+1,300)
for word,idx in word_to_idx.items():
    if word in wvmodel:
        weight[idx] = torch.from_numpy(wvmodel[word])

/tmp/ipykernel_23/3124270279.py:4: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  weight[idx] = torch.from_numpy(wvmodel[word])


## 存 pickle，下次秒载

In [19]:
import pickle
os.makedirs("pickle", exist_ok=True)
pickle.dump([train_features,train_labels_t,val_features,val_labels_t,
             test_features,weight,word_to_idx,idx_to_word,vocab],
           open("pickle/imdb_glove.pickle3",'wb'))

# Part B · 通用训练框架（只写一次，六个模型共用）

## dataloader and superX

In [20]:
from torch import nn,optim
from torch.nn import functional as F
from sklearn.metrics import accuracy_score
from tqdm import tqdm

In [21]:
batch_size,labels,lr = 64,2,0.001
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [22]:
device

device(type='cuda', index=0)

In [23]:
train_set = torch.utils.data.TensorDataset(train_features,train_labels_t)
val_set = torch.utils.data.TensorDataset(val_features,val_labels_t)
test_set = torch.utils.data.TensorDataset(test_features)
train_iter = torch.utils.data.DataLoader(train_set,batch_size,shuffle=True)
val_iter = torch.utils.data.DataLoader(val_set,batch_size,shuffle=False)
test_iter = torch.utils.data.DataLoader(test_set,batch_size,shuffle=False)

In [24]:
def train_model(net, num_epochs=5):    
# 1. 将模型移动到 GPU 或 CPU 上
    net = net.to(device)        
# 2. 定义损失函数和优化器
    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=lr)        
    for epoch in range(num_epochs):        
# ========== 训练阶段 ==========
        net.train()  # 开启训练模式（如果有 Dropout 或 BatchNorm，这会生效）
        tr_loss, tr_acc = 0.0, 0.0
        n = 0  # 【修复】：原代码中 n 未定义，这里需要初始化，用于记录训练了多少个 batch                
        for feature, label in tqdm(train_iter, desc=f'Epoch {epoch+1}/{num_epochs}'):            
# 将数据也移动到 GPU 上
            feature, label = feature.to(device), label.to(device)                        
# 梯度清零（非常重要！防止梯度累积）
            optimizer.zero_grad()                        
# 前向传播：将数据喂给模型，得到预测分数
            score = net(feature)                        
# 【修复】：原代码写成了 loss.function，正确写法是 loss_function
            loss = loss_function(score, label)                      
# 反向传播：计算梯度
            loss.backward()                        
# 更新参数：让模型根据梯度更新权重
            optimizer.step()                       
# 累加损失和准确率
            tr_loss += loss.item()
            n += 1            
# argmax(1) 表示在类别维度(第1维)上取最大值的索引，即预测类别
            tr_acc += accuracy_score(score.argmax(1).cpu(), label.cpu())        
# ========== 验证阶段 ==========
        net.eval()  # 开启评估模式（关闭 Dropout 等）
        val_acc, m = 0.0, 0                
# 验证时不需要计算梯度，使用 no_grad 可以节省显存并加速        
        with torch.no_grad():            
            for vf, vl in val_iter:
                vs = net(vf.to(device))                
# 【修复】：原代码 vs.argmax(1),cpu() 语法错误，应该是 .cpu()
                val_acc += accuracy_score(vs.argmax(1).cpu(), vl.cpu())
                m += 1                        
        print(f'Epoch {epoch+1}: Train Loss {tr_loss/n:.4f} | Train Acc {tr_acc/n:.4f} | Val Acc {val_acc/m:.4f}')        
# 【修复】：原代码 return ret，但 ret 未定义。通常我们返回训练好的模型    
    return net

In [25]:
def predict_submit(net, filename):
    net.eval()
    preds = []
    with torch.no_grad():
        for (f,) in tqdm(test_iter, desc='predict'):
            preds.extend(net(f.to(device)).argmax(1).cpu().numpy().tolist())
    sample = pd.read_csv('/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv')
    assert len(preds) == len(sample) == 25000
    assert (test.id.str.strip('"').values == sample.id.values).all()   # 顺序对齐校验
    sub = sample.copy()
    sub['sentiment'] = preds
    sub.to_csv(filename, index=False)
    print('saved:', filename)

# Part C · 模型动物园：六个类，随插随换

In [26]:
class CNNNet(nn.Module):
    def __init__(self, weight, num_filter=128, filter_size=3, labels=2):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = False   # 冻结 GloVe（迁移学习直觉）
        self.conv1d = nn.Conv1d(weight.shape[1], num_filter, filter_size, padding=1)
        self.decoder = nn.Linear(num_filter, labels)
    def forward(self, inputs):
        x = self.embedding(inputs)                     # (b,512,300)
        x = F.relu(self.conv1d(x.permute([0,2,1])))     # Conv1d 要通道在第2维
        x = F.max_pool1d(x, kernel_size=x.shape[2])
        return self.decoder(x.squeeze(2))

net = train_model(CNNNet(weight), num_epochs=20)
predict_submit(net, 'result/cnn.csv')

Epoch 1/20: 100%|██████████| 313/313 [00:03<00:00, 82.81it/s] 


Epoch 1: Train Loss 0.3732 | Train Acc 0.8347 | Val Acc 0.8768


Epoch 2/20: 100%|██████████| 313/313 [00:02<00:00, 112.42it/s]


Epoch 2: Train Loss 0.2534 | Train Acc 0.8966 | Val Acc 0.8877


Epoch 3/20: 100%|██████████| 313/313 [00:02<00:00, 110.84it/s]


...

Epoch 19: Train Loss 0.0009 | Train Acc 1.0000 | Val Acc 0.8975


Epoch 20/20: 100%|██████████| 313/313 [00:02<00:00, 106.55it/s]


Epoch 20: Train Loss 0.0007 | Train Acc 1.0000 | Val Acc 0.8977


predict: 100%|██████████| 391/391 [00:01<00:00, 302.10it/s]


saved: result/cnn.csv


In [27]:
class LSTMNet(nn.Module):
    def __init__(self,weight,num_hiddens=128,num_layers=2,labels=2):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = False
        self.encoder = nn.LSTM(input_size=weight.shape[1],hidden_size=num_hiddens,
                              num_layers=num_layers,bidirectional=True)
        self.decoder = nn.Linear(num_hiddens * 4,labels)
    def forward(self,inputs):
        x = self.embedding(inputs).permute(1,0,2)
        states,_ = self.encoder(x)
        encoding = torch.cat([states[0],states[-1]],dim=1)
        return self.decoder(encoding)
net = train_model(LSTMNet(weight), num_epochs=20)
predict_submit(net, 'result/lstm.csv')

Epoch 1/20: 100%|██████████| 313/313 [00:24<00:00, 12.95it/s]


Epoch 1: Train Loss 0.6316 | Train Acc 0.6487 | Val Acc 0.6980


Epoch 2/20: 100%|██████████| 313/313 [00:25<00:00, 12.36it/s]


Epoch 2: Train Loss 0.5523 | Train Acc 0.7191 | Val Acc 0.7417


Epoch 3/20: 100%|██████████| 313/313 [00:25<00:00, 12.33it/s]


...

Epoch 19: Train Loss 0.0576 | Train Acc 0.9819 | Val Acc 0.8879


Epoch 20/20: 100%|██████████| 313/313 [00:25<00:00, 12.35it/s]


Epoch 20: Train Loss 0.0513 | Train Acc 0.9831 | Val Acc 0.8916


predict: 100%|██████████| 391/391 [00:10<00:00, 37.83it/s]


saved: result/lstm.csv


In [28]:
class GRUNet(nn.Module):
    def __init__(self,weight,num_hiddens=128,num_layers=2,labels=2):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = False
        self.encoder = nn.GRU(input_size=weight.shape[1],hidden_size=num_hiddens,
                              num_layers=num_layers,bidirectional=True)
        self.decoder = nn.Linear(num_hiddens * 4,labels)
    def forward(self,inputs):
        x = self.embedding(inputs).permute(1,0,2)
        states,_ = self.encoder(x)
        encoding = torch.cat([states[0],states[-1]],dim=1)
        return self.decoder(encoding)
net = train_model(GRUNet(weight), num_epochs=20)
predict_submit(net, 'result/gru.csv')

Epoch 1/20: 100%|██████████| 313/313 [00:18<00:00, 16.83it/s]


Epoch 1: Train Loss 0.5230 | Train Acc 0.7238 | Val Acc 0.8740


Epoch 2/20: 100%|██████████| 313/313 [00:18<00:00, 16.56it/s]


Epoch 2: Train Loss 0.2872 | Train Acc 0.8812 | Val Acc 0.8946


Epoch 3/20: 100%|██████████| 313/313 [00:18<00:00, 16.49it/s]


...

Epoch 19: Train Loss 0.0161 | Train Acc 0.9947 | Val Acc 0.8900


Epoch 20/20: 100%|██████████| 313/313 [00:19<00:00, 16.46it/s]


Epoch 20: Train Loss 0.0152 | Train Acc 0.9947 | Val Acc 0.8942


predict: 100%|██████████| 391/391 [00:07<00:00, 55.46it/s]

saved: result/gru.csv


In [29]:
class Attention(nn.Module):
    def __init__(self, num_hiddens):
        super().__init__()
        self.w = nn.Parameter(torch.Tensor(num_hiddens * 2, num_hiddens * 2))
        self.u = nn.Parameter(torch.Tensor(num_hiddens * 2, 1))
        nn.init.uniform_(self.w, -0.1, 0.1)
        nn.init.uniform_(self.u, -0.1, 0.1)

    def forward(self, x):
        score = torch.matmul(torch.tanh(torch.matmul(x, self.w)), self.u)
        att = F.softmax(score, dim=0)
        return x * att

class AttLSTMNet(nn.Module):
    def __init__(self, weight, num_hiddens=128, num_layers=2, labels=2):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = False
        self.encoder = nn.LSTM(weight.shape[1], num_hiddens, num_layers, bidirectional=True)
        self.attention = Attention(num_hiddens)
        self.decoder = nn.Linear(num_hiddens * 2, labels)  # 改为 *2（256）

    def forward(self, inputs):
        x = self.embedding(inputs).permute([1, 0, 2])
        states, _ = self.encoder(x)
        att_out = self.attention(states)
        encoding = att_out.sum(dim=0)           # 沿时间步求和
        return self.decoder(encoding)
net = train_model(AttLSTMNet(weight), num_epochs=20)
predict_submit(net, 'result/attlstm.csv')

Epoch 1/20: 100%|██████████| 313/313 [00:28<00:00, 11.09it/s]


Epoch 1: Train Loss 0.4063 | Train Acc 0.8058 | Val Acc 0.8837


Epoch 2/20: 100%|██████████| 313/313 [00:28<00:00, 11.10it/s]


Epoch 2: Train Loss 0.2843 | Train Acc 0.8817 | Val Acc 0.8950


Epoch 3/20: 100%|██████████| 313/313 [00:28<00:00, 11.10it/s]


...

Epoch 19: Train Loss 0.0146 | Train Acc 0.9947 | Val Acc 0.8904


Epoch 20/20: 100%|██████████| 313/313 [00:27<00:00, 11.24it/s]


Epoch 20: Train Loss 0.0128 | Train Acc 0.9958 | Val Acc 0.8948


predict: 100%|██████████| 391/391 [00:11<00:00, 35.23it/s]


saved: result/attlstm.csv


In [30]:
class CNNLSTMNet(nn.Module):
    def __init__(self, weight, num_filter=128, num_hiddens=128, labels=2):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = False
        self.conv1d = nn.Conv1d(weight.shape[1], num_filter, 3, padding=1)
        self.encoder = nn.LSTM(num_filter, num_hiddens, 2, bidirectional=True)  # 注意：LSTM 输入是卷积输出的 128，不再是 300
        self.decoder = nn.Linear(num_hiddens * 4, labels)
    def forward(self, inputs):
        x = F.relu(self.conv1d(self.embedding(inputs).permute([0,2,1])))  # (b,128,512)
        states, _ = self.encoder(x.permute([2,0,1]))                      # (512,b,256)
        return self.decoder(torch.cat([states[0], states[-1]], dim=1))
net = train_model(CNNLSTMNet(weight), num_epochs=20)
predict_submit(net, 'result/cnnlstm.csv')

Epoch 1/20: 100%|██████████| 313/313 [00:23<00:00, 13.43it/s]


Epoch 1: Train Loss 0.6104 | Train Acc 0.6615 | Val Acc 0.6770


Epoch 2/20: 100%|██████████| 313/313 [00:23<00:00, 13.33it/s]


Epoch 2: Train Loss 0.6044 | Train Acc 0.6695 | Val Acc 0.8534


Epoch 3/20: 100%|██████████| 313/313 [00:23<00:00, 13.21it/s]


...

Epoch 19: Train Loss 0.0279 | Train Acc 0.9912 | Val Acc 0.8805


Epoch 20/20: 100%|██████████| 313/313 [00:23<00:00, 13.17it/s]


Epoch 20: Train Loss 0.0295 | Train Acc 0.9905 | Val Acc 0.8756


predict: 100%|██████████| 391/391 [00:10<00:00, 38.59it/s]


saved: result/cnnlstm.csv


In [31]:
class TransformerNet(nn.Module):
    def __init__(self, weight, d_model=128, nhead=8, num_layers=2, labels=2):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = False
        self.proj = nn.Linear(weight.shape[1], d_model)      # 300 → d_model，对齐维度
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=8,
                                           dim_feedforward=256, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.decoder = nn.Linear(d_model, labels)
    def forward(self, inputs):
        x = self.proj(self.embedding(inputs))    # (b,512,d_model)，batch_first=True 不用 permute
        x = self.encoder(x)                      # 自注意力：每个词和所有词交互
        return self.decoder(x.mean(dim=1))   # 对序列维平均 → (b,d_model)
net = train_model(TransformerNet(weight), num_epochs=20)
predict_submit(net, 'result/transformer.csv')

Epoch 1/20: 100%|██████████| 313/313 [00:38<00:00,  8.06it/s]


Epoch 1: Train Loss 0.4571 | Train Acc 0.7608 | Val Acc 0.8701


Epoch 2/20: 100%|██████████| 313/313 [00:38<00:00,  8.08it/s]


Epoch 2: Train Loss 0.3177 | Train Acc 0.8659 | Val Acc 0.8772


Epoch 3/20: 100%|██████████| 313/313 [00:38<00:00,  8.06it/s]


...

Epoch 19: Train Loss 0.1323 | Train Acc 0.9486 | Val Acc 0.8665


Epoch 20/20: 100%|██████████| 313/313 [00:38<00:00,  8.06it/s]


Epoch 20: Train Loss 0.1167 | Train Acc 0.9560 | Val Acc 0.8661


predict: 100%|██████████| 391/391 [00:10<00:00, 39.10it/s]


saved: result/transformer.csv
